# EONET API → SQLite  
## Notebook z Projektem bazy danych oraz importem bazy danych


<div style="text-align: center;">
    <img src="Onet.svg" width="900"  alt="Diagram bazy danych" />
</div>

### Encje (Tabele)
W schemacie występują następujące tabele:

* `event` - przechowuje podstawowe informacje o zdarzeniach (np. zjawiskach pobranych z API, z obciętym ID).
* `event_location` - przechowuje dane adresowe/lokalizacyjne (kraj, miasto) konkretnego zdarzenia.
* `geometry_type` - tabela słownikowa przechowująca typy obiektów geometrycznych.
* `geometry` - przechowuje dokładne współrzędne geograficzne, daty pomiaru oraz dodatkowe parametry (magnitude, unit).
* `category` - tabela słownikowa przechowująca kategorie zdarzeń.
* `source` - tabela słownikowa przechowująca źródła danych dla zdarzeń.
* `event_category` - tabela asocjacyjna łącząca zdarzenia z odpowiednimi kategoriami.
* `event_source` - tabela asocjacyjna łącząca zdarzenia z odpowiednimi źródłami.
* `event_geometry` - tabela asocjacyjna łącząca zdarzenia z przypisanymi im obiektami geometrycznymi.
* `import_log` - rejestr zdarzeń przechowujący informacje o procesach importu danych z API (logi).



### Lista  zależności:

| Zależność | Znaczenie |
| :--- | :--- |
| `event_location.event_id` &rarr; `event.id` | lokalizacja jest ściśle powiązana z konkretnym zdarzeniem |
| `geometry.type_id` &rarr; `geometry_type.id` | każdy rekord geometrii musi mieć określony typ (np. punkt, obszar) |
| `event` &rarr; `category` | zdarzenie jest przypisane do jedenego  lub wielu  kategorii |
| `event` &rarr; `source` | informacja o zdarzeniu pochodzi z jednego  lub  wielu źródeł |
| `event` &rarr; `geometry` | zdarzenie ma przypisane jedne  lub wiele dany współrzędnych |

## Skrypt tworzący baze danych

```sql

DROP TABLE IF EXISTS import_log;
DROP TABLE IF EXISTS event_category;
DROP TABLE IF EXISTS event_source;
DROP TABLE IF EXISTS event_geometry;
DROP TABLE IF EXISTS source;
DROP TABLE IF EXISTS category;
DROP TABLE IF EXISTS geometry;
DROP TABLE IF EXISTS geometry_type;
DROP TABLE IF EXISTS event_location;
DROP TABLE IF EXISTS event;

CREATE TABLE event (
	id INTEGER PRIMARY KEY, -- Przychodzi z API jako EONET_x zapisujemy tylko x
	title  VARCHAR(255) NOT NULL, -- Do przecinka z tego co przydchodzi w polu title
	description TEXT,
	start_date TIMESTAMP NOT NULL,
	end_date TIMESTAMP
);

CREATE TABLE event_location (
    id SERIAL PRIMARY KEY,
    event_id INTEGER NOT NULL, 
    country_code CHAR(2),
    country VARCHAR(100),
    state VARCHAR(100),
    city VARCHAR(100),

   
    CONSTRAINT fk_event_location_event
        FOREIGN KEY (event_id) 
        REFERENCES event(id)
		ON UPDATE CASCADE 
        ON DELETE CASCADE
);

CREATE TABLE geometry_type (
	id SERIAL PRIMARY KEY,
	type VARCHAR(100) NOT NULL
);

CREATE TABLE geometry (
	id SERIAL PRIMARY KEY, -- Własny z sql
	latitude FLOAT NOT NULL CHECK (90 >= latitude AND latitude >= -90), --latitude i longitude są w takich zakresach
	longitude FLOAT NOT NULL CHECK (180 >= longitude AND longitude >= -180),
	type_id INTEGER NOT NULL,
	magnitude FLOAT,
	unit FLOAT,
	measure_date TIMESTAMP NOT NULL,

	CONSTRAINT fk_geometry_type_geometry
		FOREIGN KEY (type_id)
		REFERENCES geometry_type(id)
		ON UPDATE CASCADE
		ON DELETE RESTRICT --nie chcemy usunąć typu punktu który jest zapisany w bazie
);

CREATE TABLE category (
	id SERIAL PRIMARY KEY,
	abbreviation VARCHAR(50) NOT NULL,
	title VARCHAR(255) NOT NULL,
	description TEXT
);

CREATE TABLE source (
	id SERIAL PRIMARY KEY,
	title VARCHAR(255) NOT NULL,
	description TEXT
	
);

CREATE TABLE event_category (
	event_id INTEGER NOT NULL,
	category_id INTEGER NOT NULL,

	CONSTRAINT pk_event_category
		PRIMARY KEY (event_id, category_id),
		
	CONSTRAINT fk_event_category_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_category_category
		FOREIGN KEY (category_id)
		REFERENCES category(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_source (
	event_id INTEGER  NOT NULL,
	source_id INTEGER NOT  NULL,

	
	CONSTRAINT pk_event_source
		PRIMARY KEY (event_id, source_id),

	CONSTRAINT fk_event_source_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_source_source
		FOREIGN KEY (source_id)
		REFERENCES source(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_geometry (
	event_id INTEGER NOT NULL,
	geometry_id INTEGER NOT NULL,


	CONSTRAINT pk_event_geometry
		PRIMARY KEY (event_id, geometry_id),

	CONSTRAINT fk_event_geometry_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_geometry
		FOREIGN KEY (geometry_id)
		REFERENCES geometry(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE

);

CREATE TABLE import_log (
    id SERIAL PRIMARY KEY,
    imported_at TIMESTAMP NOT NULL,
    apiname VARCHAR(255) NOT NULL,
    endpoint VARCHAR(255) NOT NULL,
    records_received INTEGER NOT NULL,
    records_saved INTEGER NOT NULL,
    status VARCHAR(100) NOT NULL,
    error_message TEXT NOT NULL
);
```

## Import bibliotek

In [1]:
from __future__ import annotations
from typing import Any
from time import sleep


from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import pandas as pd
import reverse_geocode as rg
import requests

## Tworzenie schematu bazy danych

##### Struktura pliku konfiguracyjnego (conf.txt):
DB_USER=moj_uzytkownik\
DB_PASSWORD=moje_super_tajne_haslo\
DB_HOST=localhost\
DB_PORT=5432\
DB_NAME=moja_baza_danych

In [2]:


config = {}
with open("conf.txt", "r", encoding="utf-8") as f:
    for linia in f:
        linia = linia.strip()
        if not linia or linia.startswith("#"):
            continue
        klucz, wartosc = linia.split("=", 1)
        config[klucz.strip()] = wartosc.strip()


db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=config.get("DB_USER"),
    password=config.get("DB_PASSWORD"),
    host=config.get("DB_HOST"),
    port=config.get("DB_PORT"),      
    database=config.get("DB_NAME"),
)


from scripts import schema



engine = create_engine(db_url)

try:
    conn = engine.connect()
    print("[INFO] POŁACZONO DO BAZY DANYCH")
except SQLAlchemyError as e:
    print(f"[ERROR] BŁĄD POŁĄCZENIA DO BAZY DANYCH {e}")

res = conn.execute(text(schema))
conn.commit()
print("[INFO] UTWORZONO BAZĘ DANYCH")
conn.close()

[INFO] POŁACZONO DO BAZY DANYCH
[INFO] UTWORZONO BAZĘ DANYCH


## Pobieranie danych z API


In [8]:
from  fetch_api import fetch_events, fetch_categories, fetch_source
DAYS = 100
LIMIT = 100
events = fetch_events(DAYS, LIMIT)
print(events[0:3])
sources = fetch_source()
print(sources[0:3])
categories = fetch_categories()
print(categories[0:3])

[{'id': 'EONET_20292', 'title': 'Flood in Türkiye 1103920', 'description': None, 'link': 'https://eonet.gsfc.nasa.gov/api/v3/events/EONET_20292', 'closed': '2026-06-29T00:00:00Z', 'categories': [{'id': 'floods', 'title': 'Floods'}], 'sources': [{'id': 'GDACS', 'url': 'https://www.gdacs.org/report.aspx?eventtype=FL&eventid=1103920'}], 'geometry': [{'magnitudeValue': None, 'magnitudeUnit': None, 'date': '2026-06-27T20:00:00Z', 'type': 'Point', 'coordinates': [36.4534072, 37.2401223]}]}, {'id': 'EONET_20450', 'title': 'Wildfire in Australia 1028879', 'description': None, 'link': 'https://eonet.gsfc.nasa.gov/api/v3/events/EONET_20450', 'closed': '2026-06-11T00:00:00Z', 'categories': [{'id': 'wildfires', 'title': 'Wildfires'}], 'sources': [{'id': 'GDACS', 'url': 'https://www.gdacs.org/report.aspx?eventtype=WF&eventid=1028879'}], 'geometry': [{'magnitudeValue': 5273.0, 'magnitudeUnit': 'hectare', 'date': '2026-06-11T19:00:00Z', 'type': 'Point', 'coordinates': [142.5443337537059, -12.58926091

## Zapisywanie danych do tabel

In [5]:
from sqlalchemy import text
from sqlalchemy.exc import SQLAlchemyError


def insert_event(events: list[dict], engine) -> None:

    try:
        with engine.connect() as conn:

            for event in events:

                
                if not event.get("id") or not event.get("title") or not event.get("geometry"):
                    continue

                
                event_id = int(str(event["id"]).replace("EONET_", ""))
                title = event["title"].split(",")[0].strip()
                description = event.get("description")

                start_date = event["geometry"][0]["date"]
                end_date = event.get("closed")

                conn.execute(
                    text("""
                        INSERT INTO event (
                            id, title, description, start_date, end_date
                        )
                        VALUES (
                            :id, :title, :description, :start_date, :end_date
                        )
                        ON CONFLICT (id) DO NOTHING
                    """),
                    {
                        "id": event_id,
                        "title": title,
                        "description": description,
                        "start_date": start_date,
                        "end_date": end_date
                    }
                )

                
                last_lat = None
                last_lon = None

                for geom in event["geometry"]:

                    coords = geom.get("coordinates", [None, None])
                    lon = coords[0]
                    lat = coords[1]

                    if lon is not None and lon > 180:
                        lon = lon - 360
                    
                    last_lat = lat
                    last_lon = lon

                    
                    geom_type = geom.get("type", "Point")

                    result = conn.execute(
                        text("""
                            SELECT id FROM geometry_type
                            WHERE type = :type
                        """),
                        {"type": geom_type}
                    )

                    row = result.fetchone()

                    if row:
                        type_id = row[0]
                    else:
                        type_id = conn.execute(
                            text("""
                                INSERT INTO geometry_type (type)
                                VALUES (:type)
                                RETURNING id
                            """),
                            {"type": geom_type}
                        ).scalar()

                    
                    result = conn.execute(
                        text("""
                            INSERT INTO geometry (
                                latitude, longitude, type_id,
                                magnitude, unit, measure_date
                            )
                            VALUES (
                                :lat, :lon, :type_id,
                                NULL, NULL, :date
                            )
                            RETURNING id
                        """),
                        {
                            "lat": lat,
                            "lon": lon,
                            "type_id": type_id,
                            "date": geom.get("date")
                        }
                    )

                    geometry_id = result.scalar()

                    conn.execute(
                        text("""
                            INSERT INTO event_geometry (
                                event_id, geometry_id
                            )
                            VALUES (:event_id, :geometry_id)
                            ON CONFLICT DO NOTHING
                        """),
                        {
                            "event_id": event_id,
                            "geometry_id": geometry_id
                        }
                    )

                print(f"[INFO] Zapisano event {event_id}")

            conn.commit()
            print("[INFO] Import zakończony")

    except SQLAlchemyError as e:
        print(f"[ERROR] {e}")


events = fetch_events()

import_sources(sources, engine)
import_categories(categories, engine)
insert_event(events, engine)
from  import_danych import import_sources, import_categories


[INFO] Zaimportowano 33 źródeł
[INFO] Zaimportowano 13 kategorii
[INFO] Zapisano event 20292
[INFO] Zapisano event 20450
[INFO] Zapisano event 20466
[INFO] Zapisano event 20467
[INFO] Zapisano event 20468
[INFO] Zapisano event 20465
[INFO] Zapisano event 20469
[INFO] Zapisano event 20470
[INFO] Zapisano event 20446
[INFO] Zapisano event 20445
[INFO] Zapisano event 20442
[INFO] Zapisano event 20425
[INFO] Zapisano event 20426
[INFO] Zapisano event 20451
[INFO] Zapisano event 20452
[INFO] Zapisano event 20447
[INFO] Zapisano event 20444
[INFO] Zapisano event 20396
[INFO] Zapisano event 20453
[INFO] Zapisano event 20454
[INFO] Zapisano event 20471
[INFO] Zapisano event 20472
[INFO] Zapisano event 20473
[INFO] Zapisano event 20474
[INFO] Zapisano event 20475
[INFO] Zapisano event 20443
[INFO] Zapisano event 20412
[INFO] Zapisano event 20382
[INFO] Zapisano event 20423
[INFO] Zapisano event 20449
[INFO] Zapisano event 20397
[INFO] Zapisano event 20427
[INFO] Zapisano event 20455
[INFO] Zapi

## Uruchomienie pełnego importu

## Weryfikacja zawartości bazy danych

## Analiza Danych i Wizualizacja